# Audio Fingerprinting: Chromaprint

## What is being explored?
This notebook explores **Chromaprint** as an audio fingerprinting method for video matching. Chromaprint is the algorithm behind the AcoustID audio recognition service and is purpose-built for robust audio identification.

## How does it work?
Chromaprint analyzes the chroma features of an audio signal — the distribution of energy across the 12 pitch classes — and encodes them into a compact integer array fingerprint. Because it operates on pitch content rather than raw waveform, it is robust to moderate compression, volume changes, and minor audio edits.

## Why Chromaprint over MFCC?
MFCC-based fingerprints capture timbral texture and are sensitive to volume, compression, and encoding differences. Chromaprint is specifically designed for audio matching and produces fingerprints that are directly comparable via Hamming distance, making it a more principled baseline for this task.

## Approach
Audio is extracted from each video in 4-second chunks with 1-second overlap, mirroring the visual chunking strategy in notebook 02. Raw Chromaprint fingerprints are computed per chunk, stored to JSON, and then compared across videos using Hamming distance with all-vs-all chunk matching.

In [ ]:
import os
import json
import time
import uuid
import numpy as np
import matplotlib.pyplot as plt
from pydub import AudioSegment
import acoustid
import chromaprint
import librosa
import librosa.display

# Config
SAMPLE_RATE = 44100
CHUNK_DURATION_SEC = 4
OVERLAP_SEC = 1
OUTPUT_JSON = "audio_fingerprints.json"
SAMPLE_VIDEO = "../data/downloads/scottkress_halloween_dl.mp4"  # Update to any available video
TOP_K = 5

TARGET_VIDEO = "scottkress_halloween_dl.mp4"
POSITIVE_VIDEOS = [
    "scottkress_halloween_ig.mp4",
    "scottkress_halloween_yt.mp4"
]

FOLDERS_TO_PROCESS = [
    "../data/downloads",
    "../data/processed/reels_processed",
    "../data/processed/shorts_processed"
]

In [ ]:
# --- Audio extraction and chunking ---

def extract_audio_segment(video_path, start_sec, end_sec, sample_rate=SAMPLE_RATE):
    """
    Extract a time-bounded mono audio segment from a video file.
    Returns (sample_rate, numpy array of samples).
    """
    audio = AudioSegment.from_file(video_path)
    audio = audio.set_channels(1).set_frame_rate(sample_rate)
    chunk = audio[int(start_sec * 1000):int(end_sec * 1000)]
    samples = np.array(chunk.get_array_of_samples(), dtype=np.int16)
    return sample_rate, samples

def get_audio_chunks(video_path, chunk_duration=CHUNK_DURATION_SEC, overlap=OVERLAP_SEC, sample_rate=SAMPLE_RATE):
    """
    Returns a list of audio chunks with metadata.
    Each chunk: {chunk_id, start_sec, end_sec, sample_rate, samples}
    """
    audio = AudioSegment.from_file(video_path)
    audio = audio.set_channels(1).set_frame_rate(sample_rate)
    duration_sec = len(audio) / 1000.0
    step_sec = chunk_duration - overlap

    chunks = []
    chunk_id = 0
    start_sec = 0.0

    while start_sec < duration_sec:
        end_sec = min(start_sec + chunk_duration, duration_sec)
        chunk = audio[int(start_sec * 1000):int(end_sec * 1000)]
        samples = np.array(chunk.get_array_of_samples(), dtype=np.int16)

        chunks.append({
            "chunk_id": chunk_id,
            "start_sec": round(start_sec, 3),
            "end_sec": round(end_sec, 3),
            "sample_rate": sample_rate,
            "samples": samples
        })

        chunk_id += 1
        start_sec += step_sec

    return chunks

def compute_chromaprint(samples, sample_rate=SAMPLE_RATE):
    """
    Compute raw Chromaprint fingerprint from int16 samples.
    Returns a list of integers.
    """
    duration, fp_encoded = acoustid.fingerprint(sample_rate, samples)
    fp_raw = chromaprint.decode_fingerprint(fp_encoded)[0]
    return fp_raw

print("Audio chunking and Chromaprint functions ready.")

In [ ]:
# --- Chromaprint pipeline visualization ---
# Shows waveform, spectrogram, and fingerprint heatmap for one chunk of the sample video

sr, samples = extract_audio_segment(SAMPLE_VIDEO, start_sec=0, end_sec=CHUNK_DURATION_SEC)
fp_raw = compute_chromaprint(samples, sr)

# Convert int16 samples to float for librosa
y_float = samples.astype(np.float32) / 32768.0

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Waveform
axes[0].plot(np.linspace(0, CHUNK_DURATION_SEC, len(y_float)), y_float, linewidth=0.5)
axes[0].set_title("Waveform", fontsize=10)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Spectrogram
D = librosa.amplitude_to_db(np.abs(librosa.stft(y_float)), ref=np.max)
librosa.display.specshow(D, sr=sr, x_axis="time", y_axis="log", ax=axes[1], cmap="inferno")
axes[1].set_title("Spectrogram (dB)", fontsize=10)

# Chromaprint fingerprint as heatmap
fp_array = np.array(fp_raw, dtype=np.int32)
# Reshape into 2D for visualization: unpack bits
fp_bits = np.unpackbits(fp_array.view(np.uint8)).reshape(-1, 32)[:len(fp_raw), :]
axes[2].imshow(fp_bits.T, aspect="auto", cmap="gray", interpolation="nearest")
axes[2].set_title(f"Chromaprint Fingerprint\n({len(fp_raw)} integers)", fontsize=10)
axes[2].set_xlabel("Fingerprint position")
axes[2].set_ylabel("Bit")

plt.suptitle(f"Chromaprint Pipeline — {os.path.basename(SAMPLE_VIDEO)} (0–{CHUNK_DURATION_SEC}s)", fontsize=11)
plt.tight_layout()
plt.show()

print(f"Fingerprint length: {len(fp_raw)} integers")

In [ ]:
# --- Generate and save fingerprints to JSON ---

all_data = {}
timing = {}  # {video_name: processing_seconds}
total_start = time.time()

for folder in FOLDERS_TO_PROCESS:
    folder_path = os.path.abspath(folder)
    if not os.path.exists(folder_path):
        print(f"Skipping missing folder: {folder_path}")
        continue

    for file_name in os.listdir(folder_path):
        if not file_name.lower().endswith((".mp4", ".mov", ".mkv")):
            continue

        video_path = os.path.join(folder_path, file_name)
        print(f"Processing: {file_name}")

        start_time = time.time()
        chunks = get_audio_chunks(video_path)
        video_entry = {"chunks": []}

        for chunk in chunks:
            fp_raw = compute_chromaprint(chunk["samples"], chunk["sample_rate"])
            video_entry["chunks"].append({
                "chunk_id": chunk["chunk_id"],
                "start_sec": chunk["start_sec"],
                "end_sec": chunk["end_sec"],
                "fingerprint": fp_raw
            })

        processing_time = round(time.time() - start_time, 3)
        timing[file_name] = processing_time
        all_data[file_name] = video_entry
        print(f"  Done. Chunks: {len(chunks)} | Time: {processing_time}s")

total_elapsed = round(time.time() - total_start, 3)

with open(OUTPUT_JSON, "w") as f:
    json.dump(all_data, f)

print(f"\nJSON saved to {OUTPUT_JSON}")
print(f"Total generation time: {total_elapsed}s")

In [ ]:
# --- Timing and storage metrics ---

json_size_kb = os.path.getsize(OUTPUT_JSON) / 1024

print(f"{'Video':<45} {'Time (s)':>10}")
print("-" * 57)
for video, t in timing.items():
    print(f"{video:<45} {t:>10.3f}")
print("-" * 57)
print(f"{'TOTAL':<45} {total_elapsed:>10.3f}")
print(f"\nJSON file size: {json_size_kb:.2f} KB")

In [ ]:
# --- Load JSON and comparison functions ---

with open(OUTPUT_JSON, "r") as f:
    fingerprint_data = json.load(f)

def get_chunk_fingerprints(video_name):
    """Returns list of fingerprint arrays per chunk."""
    return [
        np.array(chunk["fingerprint"], dtype=np.int32)
        for chunk in fingerprint_data[video_name]["chunks"]
    ]

def hamming_distance(fp1, fp2):
    """
    Compute normalized Hamming distance between two Chromaprint fingerprints.
    Handles different lengths by comparing only the overlapping portion.
    """
    min_len = min(len(fp1), len(fp2))
    if min_len == 0:
        return 1.0
    xor = np.bitwise_xor(fp1[:min_len], fp2[:min_len])
    bits_diff = sum(bin(x).count('1') for x in xor)
    total_bits = min_len * 32
    return bits_diff / total_bits

def compare_videos(target_chunks, other_chunks, top_k=TOP_K):
    """All-vs-all chunk comparison. Returns min_diff and mean top-k."""
    all_dists = [
        hamming_distance(c1, c2)
        for c1 in target_chunks
        for c2 in other_chunks
    ]
    if not all_dists:
        return None, None
    all_dists_sorted = sorted(all_dists)
    return all_dists_sorted[0], np.mean(all_dists_sorted[:top_k])

# Run all comparisons
all_videos = list(fingerprint_data.keys())
target_chunks = get_chunk_fingerprints(TARGET_VIDEO)
results = {}

for video in all_videos:
    if video == TARGET_VIDEO:
        continue
    other_chunks = get_chunk_fingerprints(video)
    min_diff, mean_top_k = compare_videos(target_chunks, other_chunks)
    results[video] = {"min_diff": min_diff, "mean_top_k": mean_top_k}

# Dynamic thresholds: worst positive match
pos_min_diffs = [results[v]["min_diff"] for v in POSITIVE_VIDEOS if v in results]
pos_mean_top_k = [results[v]["mean_top_k"] for v in POSITIVE_VIDEOS if v in results]
threshold_min_diff = max(pos_min_diffs)
threshold_mean_top_k = max(pos_mean_top_k)

print(f"min_diff threshold:   {threshold_min_diff:.6f}")
print(f"mean_top_k threshold: {threshold_mean_top_k:.6f}")

# Raw scores table
print(f"\n{'Video':<45} {'min_diff':>10} {'mean_top_k':>12}")
print("-" * 70)
for video, scores in sorted(results.items(), key=lambda x: x[1]['min_diff']):
    marker = " *" if video in POSITIVE_VIDEOS else ""
    print(f"{video:<45} {scores['min_diff']:>10.6f} {scores['mean_top_k']:>12.6f}{marker}")
print("\n* = positive video")

In [ ]:
# --- Confusion matrices ---

def confusion_counts(results, positive_videos, threshold, score_key):
    tp = sum(1 for v, s in results.items() if v in positive_videos and s[score_key] <= threshold)
    fp = sum(1 for v, s in results.items() if v not in positive_videos and s[score_key] <= threshold)
    tn = sum(1 for v, s in results.items() if v not in positive_videos and s[score_key] > threshold)
    fn = sum(1 for v, s in results.items() if v in positive_videos and s[score_key] > threshold)
    return tp, fp, tn, fn

def plot_confusion_matrix(ax, tp, fp, tn, fn, title):
    matrix = np.array([[tp, fn], [fp, tn]])
    ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Predicted Match", "Predicted No Match"])
    ax.set_yticklabels(["Actual Positive", "Actual Negative"])
    ax.set_title(title, fontsize=10)
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", f"TN\n{tn}"]]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, labels[i][j], ha="center", va="center", fontsize=12, color="black")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

tp, fp, tn, fn = confusion_counts(results, POSITIVE_VIDEOS, threshold_min_diff, "min_diff")
plot_confusion_matrix(axes[0], tp, fp, tn, fn,
    f"Chromaprint — min_diff (threshold: {threshold_min_diff:.6f})")

tp, fp, tn, fn = confusion_counts(results, POSITIVE_VIDEOS, threshold_mean_top_k, "mean_top_k")
plot_confusion_matrix(axes[1], tp, fp, tn, fn,
    f"Chromaprint — mean top-{TOP_K} (threshold: {threshold_mean_top_k:.6f})")

plt.suptitle(f"Confusion Matrices — Target: {TARGET_VIDEO}", fontsize=13)
plt.tight_layout()
plt.show()

## Conclusion

**Method chosen:** 

**Reasoning:** 